In [17]:
import sqlite3
import pandas as pd

# Load datasets
athletes = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/athlete_events.csv")
regions = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/noc_regions.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv", sep='\t')


# Connect to SQLite in-memory DB
conn = sqlite3.connect(":memory:")

# Write DataFrames to SQL tables
athletes.to_sql("athletes_table", conn, index=False, if_exists="replace")
regions.to_sql("regions_table", conn, index=False, if_exists="replace")
sales.to_sql("sales_table", conn, index=False, if_exists="replace")

4622

SECTION 1

Helper function to run SQL queries

In [ ]:
def run_query(title: str, sql: str) -> pd.DataFrame:
    """
    Execute a SQL query against the in-memory database and print results.
 
    Parameters
    ----------
    title : str  –> Label printed above the results.
    sql   : str  –> Valid SQLite query string.
 
    Returns
    -------
    pd.DataFrame –> Query result as a DataFrame.
    """
    print(f"\n{'─' * 60}")
    print(f"  {title}")
    print(f"{'─' * 60}")
    df = pd.read_sql_query(sql, conn)
    print(df.to_string(index=False))
    return df
 

In [ ]:
# ── INSPECT COLUMNS ───────────────────────────────────────────────────────────
# Print column names for every table so we know what dimensions we're using
 
print("\n── Column Inspector ──")
for table, df in [("athletes", athletes), ("regions", regions), ("sales", sales)]:
    print(f"\n  [{table}]  {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"  Columns: {df.columns.tolist()}")
 


── Column Inspector ──

  [athletes]  271,116 rows × 15 cols
  Columns: ['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year', 'Season', 'City', 'Sport', 'Event', 'Medal']

  [regions]  230 rows × 3 cols
  Columns: ['NOC', 'region', 'notes']

  [sales]  4,622 rows × 5 cols
  Columns: ['order_id', 'quantity', 'item_name', 'choice_description', 'item_price']


ATHLETES DATASET ////////////////////////////////////////////

In [ ]:
# ── A1. Top 5 countries by total medals ──────────────────────────────────────
run_query(
    "A1 · Top 5 countries by total medals won",
    """
    SELECT
        NOC,                        -- country code
        COUNT(Medal) AS total_medals -- COUNT ignores the NULLs, so only real medals are counted
    FROM athletes_table
    WHERE Medal IS NOT NULL         -- keep only rows where a medal was given
    GROUP BY NOC
    ORDER BY total_medals DESC
    LIMIT 5;
    """,
)


────────────────────────────────────────────────────────────
  A1 · Top 5 countries by total medals won
────────────────────────────────────────────────────────────
NOC  total_medals
USA          5637
URS          2503
GER          2165
GBR          2068
FRA          1777


,NOC,total_medals
0,USA,5637
1,URS,2503
2,GER,2165
3,GBR,2068
4,FRA,1777


In [ ]:
# ── A2. Average age of Gold medallists ───────────────────────────────────────
run_query(
    "A2 · Average age of athletes who won a Gold medal",
    """
    SELECT
        ROUND(AVG(Age), 2) AS avg_gold_age  -- round to 2 decimal places
    FROM athletes_table
    WHERE Medal = 'Gold'                     -- gold medalists only
      AND Age IS NOT NULL;                   -- exclude rows missing age
    """,
)
 


────────────────────────────────────────────────────────────
  A2 · Average age of athletes who won Gold
────────────────────────────────────────────────────────────
 avg_gold_age
         25.9


,avg_gold_age
0,25.9


In [ ]:
# ── A3. Distinct events per sport ────────────────────────────────────────────
run_query(
    "A3 · Number of distinct events in each sport",
    """
    SELECT
        Sport,
        COUNT(DISTINCT Event) AS distinct_events  --avoids double-counting same event
    FROM athletes_table
    GROUP BY Sport
    ORDER BY distinct_events DESC;
    """,
)


────────────────────────────────────────────────────────────
  A3 · Number of distinct events in each sport
────────────────────────────────────────────────────────────
                    Sport  distinct_events
                 Shooting               83
                Athletics               83
                 Swimming               55
                  Cycling               44
                  Sailing               38
                Wrestling               30
         Art Competitions               29
                  Archery               29
               Gymnastics               27
                 Canoeing               27
                   Rowing               25
     Cross Country Skiing               23
            Weightlifting               21
                  Fencing               18
            Equestrianism               18
                     Judo               15
                   Boxing               15
            Speed Skating               13
             

,Sport,distinct_events
0,Shooting,83
1,Athletics,83
2,Swimming,55
3,Cycling,44
4,Sailing,38
...,...,...
61,Cricket,1
62,Basque Pelota,1
63,Baseball,1
64,Alpinism,1


In [ ]:
# ── A4. All USA athletes ──────────────────────────────────────────────────────
run_query(
    "A4 · All athletes from the United States (NOC = 'USA')  [first 10 shown]",
    """
    SELECT
        Name, Sex, Age, Sport, Event, Medal, Year
    FROM athletes_table
    WHERE NOC = 'USA'               -- filter to usa only
    LIMIT 10;                       -- preview first 10 rows
    """,
)


────────────────────────────────────────────────────────────
  A4 · All athletes from the United States (NOC = 'USA')  [first 10 shown]
────────────────────────────────────────────────────────────
           Name Sex  Age                Sport                                               Event Medal  Year
Per Knut Aaland   M 31.0 Cross Country Skiing            Cross Country Skiing Men's 10 kilometres  None  1992
Per Knut Aaland   M 31.0 Cross Country Skiing            Cross Country Skiing Men's 50 kilometres  None  1992
Per Knut Aaland   M 31.0 Cross Country Skiing Cross Country Skiing Men's 10/15 kilometres Pursuit  None  1992
Per Knut Aaland   M 31.0 Cross Country Skiing  Cross Country Skiing Men's 4 x 10 kilometres Relay  None  1992
Per Knut Aaland   M 33.0 Cross Country Skiing            Cross Country Skiing Men's 10 kilometres  None  1994
Per Knut Aaland   M 33.0 Cross Country Skiing            Cross Country Skiing Men's 30 kilometres  None  1994
Per Knut Aaland   M 33.0 Cross C

,Name,Sex,Age,Sport,Event,Medal,Year
0,Per Knut Aaland,M,31.0,Cross Country Skiing,Cross Country Skiing Men's 10 kilometres,None,1992
1,Per Knut Aaland,M,31.0,Cross Country Skiing,Cross Country Skiing Men's 50 kilometres,None,1992
2,Per Knut Aaland,M,31.0,Cross Country Skiing,Cross Country Skiing Men's 10/15 kilometres Pu...,None,1992
3,Per Knut Aaland,M,31.0,Cross Country Skiing,Cross Country Skiing Men's 4 x 10 kilometres R...,None,1992
4,Per Knut Aaland,M,33.0,Cross Country Skiing,Cross Country Skiing Men's 10 kilometres,None,1994
5,Per Knut Aaland,M,33.0,Cross Country Skiing,Cross Country Skiing Men's 30 kilometres,None,1994
6,Per Knut Aaland,M,33.0,Cross Country Skiing,Cross Country Skiing Men's 10/15 kilometres Pu...,None,1994
7,Per Knut Aaland,M,33.0,Cross Country Skiing,Cross Country Skiing Men's 4 x 10 kilometres R...,None,1994
8,John Aalberg,M,31.0,Cross Country Skiing,Cross Country Skiing Men's 10 kilometres,None,1992
9,John Aalberg,M,31.0,Cross Country Skiing,Cross Country Skiing Men's 50 kilometres,None,1992


In [11]:
# ── A6. Athletes with missing Height OR Weight ────────────────────────────────
run_query(
    "A6 · Athlete records where Height or Weight is missing  [first 10 shown]",
    """
    SELECT
        Name, NOC, Sport, Height, Weight
    FROM athletes_table
    WHERE Height IS NULL            -- height is missing
       OR Weight IS NULL            -- OR weight is missing (either condition triggers)
    LIMIT 10;
    """,
)


────────────────────────────────────────────────────────────
  A6 · Athlete records where Height or Weight is missing  [first 10 shown]
────────────────────────────────────────────────────────────
                              Name NOC      Sport  Height Weight
               Gunnar Nielsen Aaby DEN   Football     NaN   None
              Edgar Lindenau Aabye DEN Tug-Of-War     NaN   None
Cornelia "Cor" Aalten (-Strannood) NED  Athletics   168.0   None
Cornelia "Cor" Aalten (-Strannood) NED  Athletics   168.0   None
    Einar Ferdinand "Einari" Aalto FIN   Swimming     NaN   None
              Arvo Ossian Aaltonen FIN   Swimming     NaN   None
              Arvo Ossian Aaltonen FIN   Swimming     NaN   None
              Arvo Ossian Aaltonen FIN   Swimming     NaN   None
              Arvo Ossian Aaltonen FIN   Swimming     NaN   None
              Arvo Ossian Aaltonen FIN   Swimming     NaN   None


,Name,NOC,Sport,Height,Weight
0,Gunnar Nielsen Aaby,DEN,Football,NaN,None
1,Edgar Lindenau Aabye,DEN,Tug-Of-War,NaN,None
2,"Cornelia ""Cor"" Aalten (-Strannood)",NED,Athletics,168.0,None
3,"Cornelia ""Cor"" Aalten (-Strannood)",NED,Athletics,168.0,None
4,"Einar Ferdinand ""Einari"" Aalto",FIN,Swimming,NaN,None
5,Arvo Ossian Aaltonen,FIN,Swimming,NaN,None
6,Arvo Ossian Aaltonen,FIN,Swimming,NaN,None
7,Arvo Ossian Aaltonen,FIN,Swimming,NaN,None
8,Arvo Ossian Aaltonen,FIN,Swimming,NaN,None
9,Arvo Ossian Aaltonen,FIN,Swimming,NaN,None


SALES DATASET

In [ ]:
# ── S1. Replace missing Height with average height ───────────────────────────
# I think this question was meant to be in the Athletes section, so that's what I went with
avg_height = athletes["Height"].mean()  # compute mean, pandas ignores NaN by default
 
run_query(
    f"S1 · Replace missing Height with average ({avg_height:.2f} cm) — preview",
    f"""
    SELECT
        Name,
        -- COALESCE returns the first non-NULL value
        COALESCE(Height, {avg_height:.4f}) AS Height_filled,
        Weight
    FROM athletes_table
    LIMIT 10;
    """,
)
 


────────────────────────────────────────────────────────────
  S1 · Replace missing Height with average (175.34 cm) — preview
────────────────────────────────────────────────────────────
                    Name  Height_filled  Weight
               A Dijiang        180.000    80.0
                A Lamusi        170.000    60.0
     Gunnar Nielsen Aaby        175.339     NaN
    Edgar Lindenau Aabye        175.339     NaN
Christine Jacoba Aaftink        185.000    82.0
Christine Jacoba Aaftink        185.000    82.0
Christine Jacoba Aaftink        185.000    82.0
Christine Jacoba Aaftink        185.000    82.0
Christine Jacoba Aaftink        185.000    82.0
Christine Jacoba Aaftink        185.000    82.0


,Name,Height_filled,Weight
0,A Dijiang,180.000,80.0
1,A Lamusi,170.000,60.0
2,Gunnar Nielsen Aaby,175.339,NaN
3,Edgar Lindenau Aabye,175.339,NaN
4,Christine Jacoba Aaftink,185.000,82.0
5,Christine Jacoba Aaftink,185.000,82.0
6,Christine Jacoba Aaftink,185.000,82.0
7,Christine Jacoba Aaftink,185.000,82.0
8,Christine Jacoba Aaftink,185.000,82.0
9,Christine Jacoba Aaftink,185.000,82.0


In [13]:
# ── S2. Total sales value per item ───────────────────────────────────────────
run_query(
    "S2 · Total sales value per item",
    """
    SELECT
        item_name,
        -- Strip "$" with REPLACE, cast to number, multiply by quantity ordered
        ROUND(
            SUM(CAST(REPLACE(item_price, '$', '') AS REAL) * quantity),
            2
        ) AS total_sales
    FROM sales_table
    GROUP BY item_name
    ORDER BY total_sales DESC;
    """,
)


────────────────────────────────────────────────────────────
  S2 · Total sales value per item
────────────────────────────────────────────────────────────
                            item_name  total_sales
                         Chicken Bowl      8044.63
                      Chicken Burrito      6387.06
                        Steak Burrito      4236.13
                           Steak Bowl      2479.81
                  Chips and Guacamole      2475.62
                   Chicken Salad Bowl      1506.25
                   Chicken Soft Tacos      1199.01
         Chips and Fresh Tomato Salsa      1033.96
                       Veggie Burrito      1002.27
                          Veggie Bowl       901.95
                     Barbacoa Burrito       894.75
                        Carnitas Bowl       830.71
                        Barbacoa Bowl       672.36
                        Bottled Water       649.18
                     Carnitas Burrito       616.33
                    Canned 

,item_name,total_sales
0,Chicken Bowl,8044.63
1,Chicken Burrito,6387.06
2,Steak Burrito,4236.13
3,Steak Bowl,2479.81
4,Chips and Guacamole,2475.62
5,Chicken Salad Bowl,1506.25
6,Chicken Soft Tacos,1199.01
7,Chips and Fresh Tomato Salsa,1033.96
8,Veggie Burrito,1002.27
9,Veggie Bowl,901.95


In [ ]:
# ── S3. Top 5 records by item_price ──────────────────────────────────────────
run_query(
    "S3 · Top 5 records with the highest item_price",
    """
    SELECT
        order_id,
        item_name,
        quantity,
        item_price,
        -- Numeric version of price for correct numeric sorting
        CAST(REPLACE(item_price, '$', '') AS REAL) AS price_numeric
    FROM sales_table
    ORDER BY price_numeric DESC      -- sort numerically, not alphabetically
    LIMIT 5;
    """,
)


────────────────────────────────────────────────────────────
  S3 · Top 5 records with the highest item_price
────────────────────────────────────────────────────────────
 order_id                    item_name  quantity item_price  price_numeric
     1443 Chips and Fresh Tomato Salsa        15    $44.25           44.25
     1398                Carnitas Bowl         3    $35.25           35.25
      511              Chicken Burrito         4    $35.00           35.00
     1443              Chicken Burrito         4    $35.00           35.00
     1443               Veggie Burrito         3    $33.75           33.75


,order_id,item_name,quantity,item_price,price_numeric
0,1443,Chips and Fresh Tomato Salsa,15,$44.25,44.25
1,1398,Carnitas Bowl,3,$35.25,35.25
2,511,Chicken Burrito,4,$35.00,35.00
3,1443,Chicken Burrito,4,$35.00,35.00
4,1443,Veggie Burrito,3,$33.75,33.75


In [ ]:
# ── S4. Unique customer orders ───────────────────────────────────────────────
run_query(
    "S4 · Number of unique customer orders (each order_id = one customer)",
    """
    SELECT
        COUNT(DISTINCT order_id) AS unique_orders  -- DISTINCT removes duplicate order ids
    FROM sales_table;
    """,
)
 


────────────────────────────────────────────────────────────
  S4 · Number of unique customer orders (each order_id = one customer)
────────────────────────────────────────────────────────────
 unique_orders
          1834


,unique_orders
0,1834


In [ ]:
# ── CLEANUP ───────────────────────────────────────────────────────────────────
conn.close()  # just to close the database connection when done
print("\n" + "=" * 60)
print("  All queries complete. Connection closed.")
print("=" * 60)


  All queries complete. Connection closed.


SECTION 2 ////////////////////////////////////////////////////

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SECTION 2 · High-Performing Countries
# ══════════════════════════════════════════════════════════════════════════════

run_query(
    "Section 2 · Country performance based on medalist count & average age",
    """
    -- WITH clause: pre-aggregate medalist data from the athletes table
    -- so the outer query is clean.
    WITH medalists AS (

        -- NESTED QUERY: only keep athletes who who are in a subquery of medal winners.
        -- This isolates athletes that have won AT LEAST one medal, deduplicates by
        -- athlete name and country so one person with multiple medals doesn't skew the count.
        SELECT DISTINCT
            a.Name,
            a.NOC,
            a.Age
        FROM athletes_table a
        WHERE a.Medal IS NOT NULL       -- only rows where a medal was recorded
          AND a.Age  IS NOT NULL        -- exclude athletes with unknown age
          AND a.Name IN (              -- NESTED QUERY: confirm athlete has won at least one medal
                SELECT Name
                FROM athletes_table
                WHERE Medal IS NOT NULL
          )
    )

    -- Outer query: JOIN the CTE to regions_table to get the full country name,
    -- then calculate summary stats and classify performance with CASE.
    SELECT
        r.region                            AS country,
        COUNT(m.Name)                       AS medalist_count,
        ROUND(AVG(m.Age), 2)               AS avg_age,

        -- CASE: label each country's performance tier based on average medalist age
        CASE
            WHEN AVG(m.Age) < 25        THEN 'High'    -- younger medalists → High
            WHEN AVG(m.Age) BETWEEN 25
                             AND 30     THEN 'Medium'  -- mid-age range     → Medium
            ELSE                             'Low'     -- older average     → Low
        END AS performance

    FROM medalists m

    -- JOIN: link country code in medalists CTE to the regions_table for country names
    JOIN regions_table r ON m.NOC = r.NOC

    GROUP BY r.region
    ORDER BY medalist_count DESC;       -- most successful countries first
    """,
)


────────────────────────────────────────────────────────────
  Section 2 · Country performance based on medalist count & average age
────────────────────────────────────────────────────────────
                    country  medalist_count  avg_age performance
                        USA            4618    25.09      Medium
                     Russia            3398    25.40      Medium
                    Germany            3233    25.54      Medium
                         UK            1778    27.87      Medium
                     France            1446    27.61      Medium
                      Italy            1421    26.88      Medium
                     Sweden            1380    27.97      Medium
                     Canada            1231    26.00      Medium
                  Australia            1153    25.29      Medium
                    Hungary             990    26.42      Medium
                Netherlands             928    26.40      Medium
                     Norw

,country,medalist_count,avg_age,performance
0,USA,4618,25.09,Medium
1,Russia,3398,25.40,Medium
2,Germany,3233,25.54,Medium
3,UK,1778,27.87,Medium
4,France,1446,27.61,Medium
...,...,...,...,...
129,Cyprus,1,22.00,High
130,Curacao,1,19.00,High
131,Botswana,1,18.00,High
132,Bermuda,1,25.00,Medium
